# Reproduce and inspect the methods

This notebook is a read-only walkthrough for reviewing the final CU-BEMS analysis. It traces data quality, Question A clustering, Question B forecasting, and the 60/60 sensitivity run. The rules live in `src/cubems/`; the build scripts produce the tables inspected here. The Streamlit dashboard is the presentation layer.

From the repository root, install `requirements.txt`, run `python scripts/run_all.py`, then open this notebook. No method is re-implemented here.

In [ ]:
import json
import pandas as pd
from IPython.display import display
from cubems import config

zd = pd.read_parquet(config.ZONE_DAY_PATH)
qa = config.results_dir("qa", config.MIN_HOUR_MINUTES)
qb = config.results_dir("qb", config.MIN_HOUR_MINUTES)
print(f"Zone-day rows: {len(zd):,}; analysis year: {config.YEAR}")

## 1. Data screening and completeness

`src/cubems/qc.py` flags channel-specific spikes and flatlines before aggregation. `src/cubems/aggregate.py` then accepts an hour with at least 57 valid minutes (or 60 in the strict run); incomplete zone-days are excluded rather than filled with zero. The screening outputs and accepted day counts below let a reviewer inspect the effect.

In [ ]:
qc = pd.read_parquet(config.QC_CHANNELS_PATH)
display(qc.groupby("kind", as_index=False)[["spike_minutes", "flat_minutes"]].sum())
y = zd.loc[zd["year"] == config.YEAR]
display(y.groupby("group", as_index=False)[["usable", "usable_strict"]].sum())

## 2. Question A: recurring daily electricity patterns

`src/cubems/clustering.py` clusters complete 24-hour profiles separately for C1 and C2. The selection table records cluster size, half-year stability, silhouette, and the Ward cross-check for each tested k. F1Z2 is reported separately because its 24/7 profile would form a one-zone cluster. `scripts/run_question_a.py` writes these tables.

In [ ]:
selection = pd.read_parquet(qa / "kselect.parquet")
display(selection[["group", "k", "min_zones", "half_year_ari", "silhouette", "ward_ari", "chosen", "reason"]])
energy = pd.read_parquet(qa / "kwh.parquet")
display(energy.groupby(["group", "cluster", "use"], as_index=False)["kwh"].sum())

## 3. Question B: next-day AC zone priorities

`src/cubems/forecast.py` builds only information available before the target day: zone/calendar features, earlier AC totals, and prior-day weather. Monthly expanding-window folds develop on May–August and test on September–December. `scripts/run_question_b.py` compares the seven-day benchmark, boosting, and Ridge on development folds, locks one model, then scores the final period. The decision and fold tables expose that sequence.

In [ ]:
folds = pd.read_parquet(qb / "folds.parquet")
display(folds[["stage", "test_month", "train_rows", "test_rows", "train_last"]])
decision = json.loads((qb / "decision.json").read_text())
print({key: decision[key] for key in ("chosen", "rule", "dev_mae", "dev_nmbe", "final_mae", "final_skill_vs_benchmark")})
metrics = pd.read_parquet(qb / "metrics.parquet")
display(metrics.loc[metrics["subset"].str.startswith("final"), ["subset", "model", "n", "mae", "cvrmse", "nmbe", "top5_hit_rate", "skill_vs_benchmark"]])

## 4. Sensitivity to missing minutes

Both questions are re-run using the strict 60/60 rule. The tables below show whether the selected cluster count, chosen forecast model, and final-test MAE change. `app/views/sensitivity.py` presents the same comparison in the dashboard.

In [ ]:
qa_strict = config.results_dir("qa", config.STRICT_HOUR_MINUTES)
qb_strict = config.results_dir("qb", config.STRICT_HOUR_MINUTES)
for label, folder in (("57/60", qa), ("60/60", qa_strict)):
    selected = pd.read_parquet(folder / "kselect.parquet")
    print(label, "chosen k:", selected.loc[selected["chosen"], ["group", "k"]].to_dict("records"))
for label, folder in (("57/60", qb), ("60/60", qb_strict)):
    choice = json.loads((folder / "decision.json").read_text())["chosen"]
    scored = pd.read_parquet(folder / "metrics.parquet")
    final = scored.loc[(scored["subset"] == "final (all floors)") & (scored["model"] == choice)].iloc[0]
    print(label, "model:", choice, "final MAE:", round(final["mae"], 2), "zone-days:", int(final["n"]))

## Where to inspect the implementation

- `scripts/get_data.py`: download and checksum verification.
- `scripts/build_dataset.py`: minute-to-hour and hour-to-day aggregation.
- `src/cubems/qc.py`, `aggregate.py`, `clustering.py`, `forecast.py`, `anomaly.py`: method implementations.
- `scripts/run_question_a.py`, `run_question_b.py`: full analysis and output generation.
- `tests/test_core.py`: unit checks for the key rules.
- `README.md` and `local_daily_forecast/README.md`: decisions, assumptions, and interpretation.